# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Randaadad/FlyRank-AI/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Ranked actions + reason codes

The model output is used as a directional prioritization signal rather than as an automatic decision.

Pages with higher opportunity scores are placed higher in the review queue. Each recommendation includes a reason code to make the ranking understandable to a human reviewer.

The main reason codes describe why a page may deserve attention, such as a high opportunity score, a gap between observed and expected CTR, or a combination of opportunity and available impressions.

The recommended action is to review the page manually before making any content change.

The queue is intended to answer: "Which pages should we review first, and why?"

In [ ]:
import pandas as pd

# Copy the validated model output
action_queue = results.copy()

# Rank pages by the model's opportunity score
action_queue = action_queue.sort_values(
    "predicted_probability",
    ascending=False
).reset_index(drop=True)

action_queue["rank"] = action_queue.index + 1

# Create simple reason codes
def reason_code(row):
    if row["predicted_probability"] >= 0.75:
        return "HIGH_OPPORTUNITY"
    elif row["predicted_probability"] >= 0.50:
        return "MODERATE_OPPORTUNITY"
    else:
        return "LOWER_PRIORITY"

action_queue["reason_code"] = action_queue.apply(
    reason_code,
    axis=1
)

action_queue[[
    "rank",
    "predicted_probability",
    "reason_code"
]].head(20)


### Intended use

The purpose of this playbook is to help a content or SEO reviewer prioritize pages for further investigation.

The model provides a directional opportunity score that can help decide which pages deserve attention first. A higher score means that the page is prioritized for review; it does not mean that changing the page will definitely improve CTR.

The output is intended for decision-support, not automatic content optimization.

### Limits

The model does not prove that a content change will cause an increase in CTR.

The score depends on the available data, features, target definition, and validation design.

The model should not be treated as a replacement for human judgment. Search intent, content quality, business context, and other factors may not be fully represented in the model.

The recommendations are therefore directional and should be reviewed before action.

In [ ]:
print("Intended use: directional decision-support")
print("Automatic content changes: NOT allowed")
print("Human review required: YES")

### Human review

Before acting on a recommendation, a human reviewer should check:

1. Whether the page matches the intended search intent.
2. Whether the current title and content accurately describe the page.
3. Whether the page has enough impressions or evidence to justify review.
4. Whether the model's reason code makes sense in context.
5. Whether there are business or editorial constraints that the model cannot see.
6. Whether a content change is actually appropriate.

The model should support prioritization, but the final decision remains with a human.

### No-go list

The following actions should not be automated:

- Automatically rewriting page titles.
- Automatically publishing content changes.
- Automatically deleting or replacing content.
- Treating a high model score as proof that a page needs optimization.
- Making business-critical decisions from the model score alone.
- Automatically changing pages without human review.

In [ ]:
no_go_actions = [
    "Automatic content publishing",
    "Automatic page deletion",
    "Automatic title rewriting",
    "Automatic SEO changes",
    "Business decisions based only on model score"
]

print("No-go automation rules:")
for action in no_go_actions:
    print("-", action)

### Monitoring

The recommendations should be reviewed over time because the underlying search behavior and content data can change.

Useful monitoring signals include:

- Changes in the distribution of model scores.
- Changes in CTR patterns.
- Changes in the available feature distributions.
- Missing or unexpected feature values.
- Changes in the volume of new observations.
- Declining validation performance on newer data.

### Retrain or reassessment triggers

The model should be reassessed when new labeled data becomes available or when the data distribution changes materially.

A retraining decision should be based on measured evidence rather than on a fixed schedule alone.

If recent validation results show a meaningful decline compared with the previous evaluation, the model and feature set should be reviewed before continuing to use the recommendations.

In [ ]:
score_summary = action_queue["predicted_probability"].describe()

print("Model score distribution:")
print(score_summary)

In [ ]:
print("\nMonitoring checks:")
print("- Score distribution reviewed")
print("- Missing values should be checked")
print("- Recent validation performance should be compared")
print("- Feature distribution changes should be reviewed")

### Exports

The ranked action queue is exported to `work/outputs/` so that the research paper can reuse the same recommendations generated by this notebook.

The queue contains the ranking, model score, reason code, and other available information needed for human review.

The CSV is generated by the notebook and is not treated as a production system output.

The exported queue should be regenerated whenever the validated model output changes.

In [ ]:
from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "ranked_action_queue.csv"

action_queue.to_csv(
    output_file,
    index=False
)

print(f"Exported: {output_file}")
print(f"Rows: {len(action_queue)}")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.